In [1]:
from scenic.zoo import ScenicZooEnv
from scenic.simulators.metadrive import MetaDriveSimulator
import scenic
import gymnasium as gym
import numpy as np
# from stable_baselines3 import PPO
# from stable_baselines3.ppo import MlpPolicy
# from stable_baselines3.common.monitor import Monitor
# from stable_baselines3.common.vec_env.subproc_vec_env import SubprocVecEnv
# from stable_baselines3.common.vec_env import DummyVecEnv
# from stable_baselines3.common.utils import set_random_seed
# import supersuit as ss
import os


# %%
import contextlib
import logging
import pathlib
import time
# from collections import deque
# from dataclasses import dataclass
import torch
import torch.multiprocessing as mp
import tyro
from torch import nn, optim
from train import ActorCritic

/Users/kxu/ScenicGym/src/scenic/core/errors.py:271: UserWarning: unable to install sys.excepthook to format Scenic backtraces
  warnings.warn("unable to install sys.excepthook to format Scenic backtraces")


In [ ]:
root_user = os.path.expanduser("~")
root_user

sumo_map = root_user + "/ScenicGym/assets/maps/CARLA/Town04.net.xml"
obs_space_dict = {"agent0" :  gym.spaces.Box(-0.0, 1.0 , (252,), dtype=np.float32),
                 "agent1": gym.spaces.Box(-0.0, 1.0 , (252,), dtype=np.float32)}

action_space_dict = {'agent0': gym.spaces.Box(-1.0, 1.0, (2,), np.float32),
                     'agent1': gym.spaces.Box(-1.0, 1.0, (2,), np.float32)}

model = ActorCritic(252, gym.spaces.Box(-1.0, 1.0, (2,), np.float32))
# model.load_state_dict(torch.load("models/ppo_exp_model.pth", weights_only=True))
# model.eval()
# Model class must be defined somewhere
weights = torch.load("models/new_start_point.pth", weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(weights)
model.eval()

render_and_rt = True

scenario = scenic.scenarioFromFile("exp_local.scenic",
                               model="scenic.simulators.metadrive.model",
                           mode2D=True)
env = ScenicZooEnv(scenario, 
                       MetaDriveSimulator(sumo_map=sumo_map, render=render_and_rt, real_time=render_and_rt),
                       None, 
                       max_steps=70, 
                       observation_space = obs_space_dict, 
                       action_space = action_space_dict, 
                       agents=["agent0", "agent1"])
o, _ = env.reset()
action = dict(agent0 = [0.1, 0.5], agent1=[0.1, 0.5]) # First is throttle, second is steering
with torch.no_grad():
    for episode in range(20):
        print(f"episode: {episode}")
        for i in range(2000):

            #action = dict(agent0 = [0.1 + i * 1e-5, 0.5], agent1=[0.1 + i * 1e-5, 0.5])
            for agent in ['agent0', 'agent1']:
                mean, log_std, value = model(o[agent])
                #std = log_std.exp()
                #normal = torch.distributions.Normal(mean, std)
                #x_t = normal.rsample()
                #print(f"x_t {x_t}")
                y_t = torch.tanh(mean) # i.e. argmax normal
                #print(f"y_t {y_t}")
                agent_action = y_t * model.action_scale + model.action_bias
                #print(agent_action)
                agent_action = agent_action.cpu().numpy()
                action[agent] = agent_action
                

            o, r, te, tc, info = env.step(action)
    #         print(f"STEP RECEIVED REWARD: {r}\n")
    #         print(f"observation agent 1 shappe: {o['agent0'].shape}")
    #         print(f"observation agent 2shappe: {o['agent1'].shape}")
    #         print(f"observation: {o['agent0']}")
    #         if tc:
    #             print(f"TRUNCATED")
            if te or tc:
    #             print(f"RECEIVED LAST REWARD: {r}\n")
                break
        env.reset()

    env.close()
print("We done")

episode: 0
episode: 1
episode: 2
episode: 3
episode: 4
episode: 5
episode: 6
episode: 7
episode: 8
episode: 9


In [ ]:
# root_user = os.path.expanduser("~")
# root_user

# sumo_map = root_user + "/ScenicGym/assets/maps/CARLA/Town04.net.xml"
# obs_space_dict = {"agent0" :  gym.spaces.Box(-0.0, 1.0 , (252,), dtype=np.float32),
#                  "agent1": gym.spaces.Box(-0.0, 1.0 , (252,), dtype=np.float32)}

# action_space_dict = {'agent0': gym.spaces.Box(-1.0, 1.0, (2,), np.float32),
#                      'agent1': gym.spaces.Box(-1.0, 1.0, (2,), np.float32)}
# render_and_rt = False

# scenario = scenic.scenarioFromFile("exp_local.scenic",
#                                model="scenic.simulators.metadrive.model",
#                            mode2D=True)
# env = ScenicZooEnv(scenario, 
#                        MetaDriveSimulator(sumo_map=sumo_map, render=render_and_rt, real_time=render_and_rt),
#                        None, 
#                        max_steps=70, 
#                        observation_space = obs_space_dict, 
#                        action_space = action_space_dict, 
#                        agents=["agent0", "agent1"])

# env.reset()
# action = dict(agent0 = [0.1, 0.5], agent1=[0.1, 0.5]) # First is throttle, second is steering

# for episode in range(10):
#     print(f"episode: {episode}")
#     for i in range(2000):
        
#         action = dict(agent0 = [0.1 + i * 1e-5, 0.5], agent1=[0.1 + i * 1e-5, 0.5])
        
#         o, r, te, tc, info = env.step(action)
# #         print(f"STEP RECEIVED REWARD: {r}\n")
# #         print(f"observation agent 1 shappe: {o['agent0'].shape}")
# #         print(f"observation agent 2shappe: {o['agent1'].shape}")
# #         print(f"observation: {o['agent0']}")
# #         if tc:
# #             print(f"TRUNCATED")
#         if te:
# #             print(f"RECEIVED LAST REWARD: {r}\n")
#             break
#     env.reset()
        
# env.close()
# print("We done")

# 